# Phase 1 - Step 3: Data Cleaning

This notebook cleans all raw datasets adhering strictly to the rule: NEVER modify `data/raw/`. Cleaned outputs are saved to `data/processed/`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_dir = Path("data/raw")
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

cleaning_log = []

def log_clean(dataset, action, before_shape, after_shape, notes):
    cleaning_log.append({
        "dataset": dataset,
        "action": action,
        "before_shape": str(before_shape),
        "after_shape": str(after_shape),
        "notes": notes
    })
    print(f"[{dataset}] {action}: {notes}")


In [2]:
# 1. Clean employee_attrition.csv
df_attr = pd.read_csv(raw_dir / "employee_attrition.csv")
b_shape = df_attr.shape

# Strip whitespace from column names and string values
df_attr.columns = [c.strip() for c in df_attr.columns]
for c in df_attr.select_dtypes(include=['object', 'string']).columns:
    df_attr[c] = df_attr[c].astype(str).str.strip()

# Deduplicate
df_attr = df_attr.drop_duplicates()

# Handle CustomerSatisfaction missing values: impute median and add missing indicator
med_sat = df_attr['CustomerSatisfaction'].replace('nan', np.nan).astype(float).median()
df_attr['CustomerSatisfaction_missing'] = df_attr['CustomerSatisfaction'].replace('nan', np.nan).isnull().astype(int)
df_attr['CustomerSatisfaction'] = df_attr['CustomerSatisfaction'].replace('nan', np.nan).astype(float).fillna(med_sat)

# Ensure numeric types
num_cols = ['Age', 'EducationLevel', 'MonthlySalary', 'OvertimeHoursPerMonth', 'LeavesTaken', 
            'ProjectsHandled', 'TrainingHours', 'LastPromotionYear', 'YearsAtCompany', 'PerformanceRating']
for c in num_cols:
    df_attr[c] = pd.to_numeric(df_attr[c], errors='coerce')

df_attr['WorkLifeBalanceScore'] = pd.to_numeric(df_attr['WorkLifeBalanceScore'], errors='coerce')

attr_proc_path = processed_dir / "employee_attrition_processed.csv"
df_attr.to_csv(attr_proc_path, index=False)
log_clean("employee_attrition.csv", "Clean & Impute", b_shape, df_attr.shape, 
          f"Imputed CustomerSatisfaction with median ({med_sat}), normalized strings, saved to {attr_proc_path.name}")


[employee_attrition.csv] Clean & Impute: Imputed CustomerSatisfaction with median (5.0), normalized strings, saved to employee_attrition_processed.csv


In [3]:
# 2. Clean hr_performance_engagement.csv
df_eng = pd.read_csv(raw_dir / "hr_performance_engagement.csv")
b_shape = df_eng.shape

# Standardize columns (strip spaces, use snake_case for consistency while keeping semantic names)
df_eng.columns = [c.strip().replace(" ", "_").replace("(%)", "pct").lower() for c in df_eng.columns]
for c in df_eng.select_dtypes(include=['object', 'string']).columns:
    df_eng[c] = df_eng[c].astype(str).str.strip()

df_eng = df_eng.drop_duplicates()

# Safe numeric casting
eng_num_cols = ['performance_score', 'kpi_score', 'attendance_pct', 'peer_rating', 
                'task_completion_pct', 'work_hours_logged', 'manager_feedback', 'training_hours']
for c in eng_num_cols:
    df_eng[c] = pd.to_numeric(df_eng[c], errors='coerce')

eng_proc_path = processed_dir / "engagement_processed.csv"
df_eng.to_csv(eng_proc_path, index=False)
log_clean("hr_performance_engagement.csv", "Standardize & Clean", b_shape, df_eng.shape, 
          f"Standardized column names, typed numeric fields, saved to {eng_proc_path.name}")


[hr_performance_engagement.csv] Standardize & Clean: Standardized column names, typed numeric fields, saved to engagement_processed.csv


In [4]:
# 3. Clean occupation_data.csv
df_occ = pd.read_csv(raw_dir / "occupation_data.csv")
b_shape = df_occ.shape

df_occ.columns = [c.strip() for c in df_occ.columns]
df_occ['O*NET-SOC Code'] = df_occ['O*NET-SOC Code'].astype(str).str.strip()
df_occ['Title'] = df_occ['Title'].astype(str).str.strip()
df_occ['Description'] = df_occ['Description'].astype(str).str.strip()
df_occ = df_occ.drop_duplicates(subset=['O*NET-SOC Code'])

occ_proc_path = processed_dir / "occupation_master.csv"
df_occ.to_csv(occ_proc_path, index=False)
log_clean("occupation_data.csv", "Clean Master", b_shape, df_occ.shape, 
          f"Verified unique ONET codes, trimmed strings, saved to {occ_proc_path.name}")


[occupation_data.csv] Clean Master: Verified unique ONET codes, trimmed strings, saved to occupation_master.csv


In [5]:
# 4. Clean essential_skills.csv
df_ess = pd.read_csv(raw_dir / "essential_skills.csv")
b_shape = df_ess.shape

df_ess.columns = [c.strip() for c in df_ess.columns]
for c in ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name']:
    df_ess[c] = df_ess[c].astype(str).str.strip()

# Standardize Element Name
df_ess['Element Name'] = df_ess['Element Name'].str.title()
df_ess['Data Value'] = pd.to_numeric(df_ess['Data Value'], errors='coerce')
df_ess = df_ess.drop_duplicates()

ess_proc_path = processed_dir / "essential_skills_processed.csv"
df_ess.to_csv(ess_proc_path, index=False)
log_clean("essential_skills.csv", "Clean Essential Skills", b_shape, df_ess.shape, 
          f"Normalized skill names, parsed Data Value, saved to {ess_proc_path.name}")


[essential_skills.csv] Clean Essential Skills: Normalized skill names, parsed Data Value, saved to essential_skills_processed.csv


In [6]:
# 5. Clean software_skills.csv
df_soft = pd.read_csv(raw_dir / "software_skills.csv")
b_shape = df_soft.shape

df_soft.columns = [c.strip() for c in df_soft.columns]
for c in df_soft.select_dtypes(include=['object', 'string']).columns:
    df_soft[c] = df_soft[c].astype(str).str.strip()

# Standardize skill / software name
df_soft['Workplace Example'] = df_soft['Workplace Example'].str.strip()
df_soft['Hot Technology'] = df_soft['Hot Technology'].str.upper()
df_soft['In Demand'] = df_soft['In Demand'].str.upper()
df_soft = df_soft.drop_duplicates()

soft_proc_path = processed_dir / "software_skills_processed.csv"
df_soft.to_csv(soft_proc_path, index=False)
log_clean("software_skills.csv", "Clean Software Skills", b_shape, df_soft.shape, 
          f"Normalized software names, standardized flags, saved to {soft_proc_path.name}")


[software_skills.csv] Clean Software Skills: Normalized software names, standardized flags, saved to software_skills_processed.csv


In [7]:
clean_summary_df = pd.DataFrame(cleaning_log)
clean_summary_path = processed_dir / "data_cleaning_summary.csv"
clean_summary_df.to_csv(clean_summary_path, index=False)
print(f"\nSaved data cleaning summary to {clean_summary_path}")
print(clean_summary_df.to_string())



Saved data cleaning summary to data\processed\data_cleaning_summary.csv
                         dataset                  action before_shape  after_shape                                                                                                          notes
0         employee_attrition.csv          Clean & Impute    (500, 24)    (500, 25)  Imputed CustomerSatisfaction with median (5.0), normalized strings, saved to employee_attrition_processed.csv
1  hr_performance_engagement.csv     Standardize & Clean   (5000, 13)   (5000, 13)                             Standardized column names, typed numeric fields, saved to engagement_processed.csv
2            occupation_data.csv            Clean Master    (1016, 3)    (1016, 3)                                    Verified unique ONET codes, trimmed strings, saved to occupation_master.csv
3           essential_skills.csv  Clean Essential Skills  (18200, 15)  (18200, 15)                             Normalized skill names, parsed Data Valu